# Genre Classification of Song Lyrics with a Support Vector Machine

**Course project — Natural Language Processing & Text Analytics**

In this notebook we train a Support Vector Machine (SVM) to predict the **parent genre** of a song from its **lyrics**. The pipeline is the canonical one used in NLP text classification:

1. Load and inspect the data.
2. Split into train / test (stratified, because the classes are imbalanced).
3. Convert text into numeric features with **TF-IDF** (Term Frequency – Inverse Document Frequency).
4. Train a **Linear Support Vector Classifier (`LinearSVC`)**.
5. Evaluate with accuracy, a per-class classification report, and a confusion matrix.
6. Inspect the top words per class (interpretability) and try the model on a custom lyric.

### Why SVM for text?

Text data, after TF-IDF, lives in a very **high-dimensional, sparse space** (tens of thousands of features, mostly zero per document). SVMs — especially the linear variant — are well suited to this regime: they find a maximum-margin separating hyperplane and don't suffer from the curse of dimensionality the way distance-based models (k-NN) do. `LinearSVC` is also dramatically faster than `SVC(kernel='linear')` on this scale because it uses the LIBLINEAR solver instead of LIBSVM.

### Why TF-IDF?

Raw word counts overweight common words. TF-IDF down-weights words that appear in many documents (low information) and up-weights words that are characteristic of a few documents — exactly what we want for distinguishing genres.

## 1. Imports

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

pd.set_option("display.max_colwidth", 80)
RANDOM_STATE = 42

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 2. Load the dataset

The file `dataset.csv` contains 15,434 songs with the following columns:

- `track_id` — Spotify ID (not used for modeling)
- `lyrics` — raw lyrics
- `lyrics_cleaned` — lowercased / lightly preprocessed lyrics (we use this)
- `track_genre` — fine-grained genre (88 classes — too sparse for our sample size)
- `parent_genre` — coarse genre (13 classes — what we predict)

In [ ]:
DATA_PATH = "dataset.csv"  # place the file next to this notebook
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head(3)

## 3. Inspect the class distribution

Class balance matters because SVMs (like most classifiers) will happily sacrifice rare classes to maximize overall accuracy if you don't push back. We check the distribution first so we know what we're up against.

In [ ]:
counts = df["parent_genre"].value_counts()
print(counts)

fig, ax = plt.subplots(figsize=(9, 4))
counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Songs per parent genre")
ax.set_ylabel("Number of songs")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(f"\nImbalance ratio (largest / smallest class): {counts.max() / counts.min():.1f}x")

## 4. Train / test split (stratified)

We use an 80 / 20 split. Because of the imbalance we **stratify on the label** so every class shows up in both splits in the same proportion — without this, rare classes (e.g., Latin with 62 samples) could end up almost entirely in one split and the test results would be meaningless.

> Always fit the vectorizer **only on the training data**. Fitting on the full dataset would leak information from the test set into the model and inflate your reported accuracy.

In [ ]:
X = df["lyrics_cleaned"].astype(str)
y = df["parent_genre"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train: {len(X_train):,} songs")
print(f"Test : {len(X_test):,} songs")

## 5. TF-IDF vectorization

Hyperparameter choices and why:

- **`ngram_range=(1, 2)`** — include unigrams *and* bigrams. Bigrams capture short phrases ("baby girl", "hold on") that often carry genre signal.
- **`min_df=5`** — drop words that appear in fewer than 5 documents. These are mostly typos / rare words / proper nouns and just add noise + memory cost.
- **`max_df=0.9`** — drop words that appear in more than 90% of songs. These are filler words with no discriminative power.
- **`sublinear_tf=True`** — replace raw term frequency `tf` with `1 + log(tf)`. Common with SVMs because lyric repetition (think a chorus) shouldn't dominate the score linearly.
- **No stop-word removal** — IDF already down-weights common words, and for music a word like *"don't"* or *"the"* may still carry style. Letting TF-IDF decide is more principled than a hand-coded list.

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
    sublinear_tf=True,
)

t0 = time.time()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)
print(f"Vectorization took {time.time() - t0:.1f}s")
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,} features")
print(f"X_train shape:   {X_train_tfidf.shape}")
sparsity = 100 * (1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]))
print(f"Sparsity:        {sparsity:.2f}% zeros")

## 6. Train the SVM

We use `LinearSVC`, which fits a linear SVM via LIBLINEAR. Two important choices:

- **`class_weight="balanced"`** — automatically up-weights rare classes inversely to their frequency. Without this, the model would barely ever predict Latin or Hip-Hop & R&B.
- **`C=1.0`** — the regularization strength (the default). Lower C = more regularization (smoother decision boundary, more bias). For a course project this is a fine starting point; in a real project you would tune it with `GridSearchCV`.

In [ ]:
clf = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    max_iter=2000,
)

t0 = time.time()
clf.fit(X_train_tfidf, y_train)
print(f"Training took {time.time() - t0:.1f}s")

## 7. Evaluate

We look at three things:

1. **Overall accuracy** — single number, easy to report but misleading on imbalanced data.
2. **Per-class precision / recall / F1** — tells us *which* classes the model actually learns.
3. **Confusion matrix** — shows which classes get mistaken for which (e.g., does Rock leak into Metal?).

In [ ]:
y_pred = clf.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")
print()
print("Per-class report:")
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")

fig, ax = plt.subplots(figsize=(9, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, values_format=".2f", colorbar=False)
ax.set_title("Confusion matrix (row-normalized: P(predicted | true))")
plt.tight_layout()
plt.show()

## 8. What is the model actually learning?

Because `LinearSVC` is linear, its **coefficients are directly interpretable**: for each class, the words with the largest positive coefficients are the words that most push a document toward that class. This is one of the biggest practical advantages of linear SVM over a deep model — you can sanity-check what it learned.

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
top_n = 12

for idx, cls in enumerate(clf.classes_):
    top_idx = np.argsort(clf.coef_[idx])[-top_n:][::-1]
    print(f"{cls:>20s}: " + ", ".join(feature_names[top_idx]))

## 9. Try it on a custom lyric

You can paste any lyric here and see what the model predicts. Note: `LinearSVC` doesn't produce calibrated probabilities by default (no `predict_proba`); the raw decision-function score is the signed distance from the hyperplane — bigger means more confident.

In [ ]:
sample_lyric = """
yeah uh check it i grew up in the streets where the concrete bleeds
got my homies on the block we just chasing them dreams
hustlin all night gotta get this paper right
microphone in my hand spittin fire all night
"""

vec = vectorizer.transform([sample_lyric])
pred = clf.predict(vec)[0]
scores = clf.decision_function(vec)[0]
ranked = sorted(zip(clf.classes_, scores), key=lambda x: -x[1])

print(f"Predicted parent genre: {pred}\n")
print("Top 5 classes by decision score:")
for cls, s in ranked[:5]:
    print(f"  {cls:>20s}  {s:+.3f}")

## 10. What you would do next (if extending this)

This baseline is honest and reproducible, but there is room to improve:

- **Hyperparameter tuning** — grid-search `C`, `min_df`, `ngram_range` with `GridSearchCV` and stratified CV.
- **Better text cleaning** — strip section markers like `[Chorus]`, handle non-English lyrics (which currently get mostly mis-classified).
- **Address imbalance more aggressively** — under-sample Electronic, or use `imbalanced-learn`'s SMOTE on the TF-IDF vectors.
- **Compare models** — Logistic Regression and Multinomial Naive Bayes are common baselines; report all three.
- **Char n-grams** — `analyzer="char_wb", ngram_range=(3,5)` is surprisingly strong on lyrics with slang.
- **Move to embeddings** — sentence-transformer embeddings + `LinearSVC` often beats TF-IDF, at the cost of explainability.

Stop here for the assignment baseline; the items above are how you'd push it further.